# Heart Disease Prediction - Final Assignment\n
\n
Diploma in Data Science & Machine Learning\n
\n
This notebook demonstrates a complete end-to-end machine learning and deep learning workflow for heart disease prediction.

## 1. Import Libraries

In [ ]:
import warnings\n
from pathlib import Path\n
\n
import matplotlib.pyplot as plt\n
import numpy as np\n
import pandas as pd\n
import seaborn as sns\n
from sklearn.model_selection import train_test_split\n
\n
from src.config import TARGET_COL, RANDOM_STATE\n
from src.data_loader import load_dataset\n
from src.train_and_evaluate import (\n
    preprocess_input,\n
    train_ml_models,\n
    train_ann,\n
    save_history_plot,\n
    ensure_output_dirs,\n
)\n
\n
warnings.filterwarnings("ignore")\n
sns.set_theme(style="whitegrid")\n
np.random.seed(RANDOM_STATE)

## 2. Load Dataset

In [ ]:
ensure_output_dirs()\n
df = load_dataset()\n
df = preprocess_input(df)\n
df.head()

In [ ]:
print("Shape:", df.shape)\n
print("Missing Values:\n", df.isna().sum())\n
print("Class Distribution:\n", df[TARGET_COL].value_counts(normalize=True))

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))\n
numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]\n
for ax, col in zip(axes.flat, numeric_cols):\n
    sns.histplot(df[col], kde=True, ax=ax)\n
    ax.set_title(f"Distribution of {col}")\n
plt.tight_layout()\n
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))\n
corr = df.corr(numeric_only=True)\n
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")\n
plt.title("Correlation Heatmap")\n
plt.tight_layout()\n
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))\n
sns.countplot(data=df, x=TARGET_COL, palette="Set2")\n
plt.title("Target Class Distribution")\n
plt.xlabel("Heart Disease (0=No, 1=Yes)")\n
plt.tight_layout()\n
plt.show()

## 4. Train/Test Split

In [ ]:
X = df.drop(columns=[TARGET_COL])\n
y = df[TARGET_COL]\n
\n
X_train, X_test, y_train, y_test = train_test_split(\n
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y\n
)\n
\n
print("Training shape:", X_train.shape)\n
print("Testing shape:", X_test.shape)

## 5. Train Machine Learning Models (with Hyperparameter Tuning)

In [ ]:
ml_summary, ml_details = train_ml_models(X_train, X_test, y_train, y_test)\n
ml_summary

## 6. Train Deep Learning Model (ANN)

In [ ]:
ann_metrics, ann_history = train_ann(X_train, X_test, y_train, y_test)\n
save_history_plot(ann_history)\n
ann_metrics

## 7. Consolidated Model Comparison

In [ ]:
ann_row = pd.DataFrame([\n
    {\n
        "model": ann_metrics["model"],\n
        "accuracy": ann_metrics["accuracy"],\n
        "precision": ann_metrics["precision"],\n
        "recall": ann_metrics["recall"],\n
        "f1_score": ann_metrics["f1_score"],\n
        "roc_auc": ann_metrics["roc_auc"],\n
    }\n
])\n
\n
comparison = pd.concat([ml_summary, ann_row], ignore_index=True).sort_values("roc_auc", ascending=False)\n
comparison

In [ ]:
best_model = comparison.iloc[0]["model"]\n
print(f"Best model based on ROC-AUC: {best_model}")

## 8. View Saved Artifacts

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()\n
print("Models:")\n
for p in sorted((project_root / "models").glob("*")):\n
    print(" -", p.name)\n
\n
print(\n"Reports:")\n
for p in sorted((project_root / "reports").glob("*")):\n
    print(" -", p.name)

## 9. Conclusion\n
\n
This notebook completes the project workflow:\n
- Data loading and quality checks\n
- EDA and visualization\n
- Preprocessing and feature transformation\n
- Model training (ML + ANN)\n
- Hyperparameter tuning\n
- Model evaluation and comparison\n
- Artifact saving for deployment/inference